# 1. RBAC and Custom Roles

AZ-500 expects you to **implement** RBAC — not just know what it is. You need to create custom roles, assign them at the right scope, and understand the permission model.

## RBAC essentials

Azure RBAC has four elements:

| Element | What it is | Example |
|---------|-----------|----------|
| **Security principal** | Who gets access | User, group, service principal, managed identity |
| **Role definition** | What they can do | Actions, NotActions, DataActions, NotDataActions |
| **Scope** | Where it applies | Management group → Subscription → Resource group → Resource |
| **Assignment** | Binding principal + role + scope | "Alice is Contributor on rg-prod" |

In [ ]:
import json

# Azure RBAC scope hierarchy
SCOPE_HIERARCHY = {
    'Management Group: Contoso': {
        'Subscription: Production': {
            'Resource Group: rg-web': ['App Service: web-app', 'SQL Database: web-db', 'Key Vault: web-kv'],
            'Resource Group: rg-data': ['Storage Account: datalake', 'Synapse Workspace: analytics'],
        },
        'Subscription: Development': {
            'Resource Group: rg-dev': ['App Service: dev-app', 'SQL Database: dev-db'],
        },
    },
}

def show_scope(scope, indent=0):
    prefix = '  ' * indent
    if isinstance(scope, dict):
        for k, v in scope.items():
            print(f'{prefix}📁 {k}')
            show_scope(v, indent + 1)
    elif isinstance(scope, list):
        for item in scope:
            print(f'{prefix}📄 {item}')

print('=== Azure RBAC Scope Hierarchy ===')
print('Roles assigned at a higher scope are INHERITED by all children.\n')
show_scope(SCOPE_HIERARCHY)
print('\n💡 If Alice is "Contributor" on the Production subscription,')
print('   she has Contributor access to EVERY resource in rg-web and rg-data.')

## Built-in roles

Azure has 300+ built-in roles. Key ones for the exam:

| Role | Permissions | Exam scenario |
|------|-----------|----------------|
| **Owner** | Everything + assign roles | Subscription admins |
| **Contributor** | Everything except assign roles | DevOps teams |
| **Reader** | Read-only | Auditors |
| **User Access Administrator** | Manage role assignments only | Delegating access management |
| **Security Admin** | Manage Defender for Cloud + security policies | SOC teams |
| **Security Reader** | Read security data | Compliance reviewers |
| **Key Vault Administrator** | Full Key Vault management | Ops teams |
| **Key Vault Secrets User** | Read secrets only | Applications |
| **Storage Blob Data Contributor** | Read/write blob data (data plane) | Applications accessing storage |
| **Network Contributor** | Manage networking resources | Network team |

### Control plane vs data plane

This distinction is critical for AZ-500:

- **Control plane** (`Actions`/`NotActions`): manage the resource itself (create, delete, configure).
- **Data plane** (`DataActions`/`NotDataActions`): access the data inside the resource (read blobs, get secrets).

**Contributor** has full control-plane access but **zero** data-plane access to Key Vault or Storage. This is why you need `Key Vault Secrets User` or `Storage Blob Data Contributor` separately.

In [ ]:
# Simulate role definitions with Actions/DataActions
ROLES = {
    'Contributor': {
        'Actions': ['*'],
        'NotActions': ['Microsoft.Authorization/roleAssignments/*', 'Microsoft.Authorization/roleDefinitions/*'],
        'DataActions': [],
        'NotDataActions': [],
    },
    'Storage Blob Data Contributor': {
        'Actions': ['Microsoft.Storage/storageAccounts/blobServices/containers/delete',
                    'Microsoft.Storage/storageAccounts/blobServices/containers/read',
                    'Microsoft.Storage/storageAccounts/blobServices/containers/write'],
        'DataActions': ['Microsoft.Storage/storageAccounts/blobServices/containers/blobs/*'],
        'NotDataActions': [],
    },
    'Key Vault Secrets User': {
        'Actions': [],
        'DataActions': ['Microsoft.KeyVault/vaults/secrets/getSecret/action',
                        'Microsoft.KeyVault/vaults/secrets/readMetadata/action'],
        'NotDataActions': [],
    },
}

def check_permission(role_name: str, operation: str, plane: str) -> str:
    role = ROLES.get(role_name)
    if plane == 'control':
        actions, not_actions = role['Actions'], role['NotActions']
    else:
        actions, not_actions = role['DataActions'], role['NotDataActions']
    
    denied = any(operation.startswith(na.replace('/*', '/')) or na == operation for na in not_actions)
    allowed = any(a == '*' or operation.startswith(a.replace('/*', '/')) for a in actions)
    
    if denied:
        return '❌ DENIED (NotActions)'
    if allowed:
        return '✅ ALLOWED'
    return '❌ NOT PERMITTED (no matching action)'

print('=== Permission checks ===\n')
checks = [
    ('Contributor', 'Microsoft.Compute/virtualMachines/delete', 'control', 'Delete a VM'),
    ('Contributor', 'Microsoft.Authorization/roleAssignments/write', 'control', 'Assign a role'),
    ('Contributor', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('Key Vault Secrets User', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('Storage Blob Data Contributor', 'Microsoft.Storage/storageAccounts/blobServices/containers/blobs/read', 'data', 'Read a blob'),
]

for role, op, plane, desc in checks:
    result = check_permission(role, op, plane)
    print(f'{result}  {role:<35} → {desc}')

## Custom roles

When built-in roles don't fit, create a custom role. Common exam scenario: *"The VM support team needs to restart VMs but not delete them."*

### Custom role JSON definition

In [ ]:
custom_role = {
    'Name': 'VM Restart Operator',
    'Description': 'Can view and restart virtual machines, but not delete or create them.',
    'Actions': [
        'Microsoft.Compute/virtualMachines/read',
        'Microsoft.Compute/virtualMachines/restart/action',
        'Microsoft.Compute/virtualMachines/start/action',
        'Microsoft.Compute/virtualMachines/powerOff/action',
        'Microsoft.Resources/subscriptions/resourceGroups/read',
    ],
    'NotActions': [],
    'DataActions': [],
    'NotDataActions': [],
    'AssignableScopes': ['/subscriptions/00000000-0000-0000-0000-000000000000'],
}

print('Custom role definition:')
print(json.dumps(custom_role, indent=2))

print('\n--- Azure CLI to create this role ---')
print('az role definition create --role-definition @vm-restart-operator.json')

print('\n--- Azure CLI to assign it ---')
print('az role assignment create \\')
print('  --assignee alice@contoso.com \\')
print('  --role "VM Restart Operator" \\')
print('  --scope /subscriptions/.../resourceGroups/rg-prod')

# Verify
print('\n--- What Alice can and cannot do ---')
test_ops = [
    ('Microsoft.Compute/virtualMachines/read', 'View VMs'),
    ('Microsoft.Compute/virtualMachines/restart/action', 'Restart VMs'),
    ('Microsoft.Compute/virtualMachines/delete', 'Delete VMs'),
    ('Microsoft.Compute/virtualMachines/write', 'Create/update VMs'),
]
for op, desc in test_ops:
    allowed = any(op.startswith(a.replace('/action', '')) or a == op for a in custom_role['Actions'])
    print(f'  {"✅" if allowed else "❌"} {desc}')

### Custom role limits

- Max **5,000 custom roles** per tenant (Azure AD) or per subscription.
- `AssignableScopes` must include at least one management group, subscription, or resource group.
- Custom roles can only be used within their assignable scopes.
- Takes up to **5 minutes** for role definition changes to propagate.

### Exam tip: deny assignments

Azure also supports **deny assignments** that block access even if a role grants it. These are created by Azure Blueprints and managed apps — you can't create them directly. Deny always wins over allow.

---
## Summary

| Concept | What to implement |
|---------|-------------------|
| **Scope hierarchy** | MG → Sub → RG → Resource. Roles inherit downward. |
| **Control vs data plane** | `Actions` = manage the resource. `DataActions` = access the data inside. |
| **Contributor gap** | Full control-plane, zero data-plane. Need separate data roles. |
| **Custom roles** | JSON with Actions/NotActions/DataActions. Use narrowest permissions possible. |
| **Deny assignments** | Always win. Created by Blueprints, not by users. |

**Next**: [Notebook 2 — PIM and Conditional Access](02_pim_and_conditional_access.ipynb)